<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/tpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import jax

print("Backend:", jax.default_backend())
print("Devices:", jax.devices())

/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:90: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Backend: tpu
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


In [2]:
import time
import jax
import jax.numpy as jnp

print("Backend:", jax.default_backend())
print("Devices:", jax.devices())

if jax.default_backend() != "tpu":
    raise RuntimeError("Select TPU under Runtime → Change runtime type first.")

@jax.jit
def matrix_multiply(a, b):
    return a @ b

def benchmark(size, repeats=10):
    key_a, key_b = jax.random.split(jax.random.PRNGKey(42))
    a = jax.random.normal(key_a, (size, size), dtype=jnp.bfloat16)
    b = jax.random.normal(key_b, (size, size), dtype=jnp.bfloat16)


    result = matrix_multiply(a, b)
    result.block_until_ready()

    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        result = matrix_multiply(a, b)
        result.block_until_ready()
        times.append(time.perf_counter() - start)

    median_seconds = sorted(times)[len(times) // 2]
    operations = 2 * size**3
    achieved_tflops = operations / median_seconds / 1e12

    print(
        f"{size} × {size} | "
        f"median: {median_seconds * 1000:.2f} ms | "
        f"approx. {achieved_tflops:.2f} TFLOP/s"
    )

for size in [1024, 2048, 4096]:
    benchmark(size)

Backend: tpu
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
1024 × 1024 | median: 0.25 ms | approx. 8.65 TFLOP/s
2048 × 2048 | median: 0.29 ms | approx. 59.32 TFLOP/s
4096 × 4096 | median: 0.99 ms | approx. 139.20 TFLOP/s
